# Import Libraries

In [ ]:
!pip install torch_geometric -q

In [ ]:
import torch
# from torch_geometric.datasets import AMiner, Taobao, MovieLens1M, AmazonBook, HM
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import time
import psutil
import gc
import sys

In [ ]:
!pip install cython
!pip install networkit

In [ ]:
import networkit as nk
from scipy.sparse import coo_matrix

# Download and Preprocess Data

## Flickr

In [ ]:
from torch_geometric.datasets import Flickr

In [ ]:
Flickr_data = Flickr(root='data/Flickr')

In [ ]:
Flickr_data.data.x.shape

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


torch.Size([89250, 500])

In [ ]:
is_row_zero = (Flickr_data.data.x == 0).all(dim=1)
# Flickr_data.data.x[is_row_zero]

In [ ]:
# num_rows = 22312
# # Step 1: Preprocess and select top num_rows
# sub_node_vectors = Flickr_data.data.x[~is_row_zero][:num_rows, :].numpy()  # Converting to NumPy for processing

# # Step 2: Compute the adjacency matrix (Cosine similarity)
# norms = np.linalg.norm(sub_node_vectors, axis=1)
# adjacency_matrix = np.dot(sub_node_vectors, sub_node_vectors.T) / (norms[:, None] * norms[None, :])

# # Normalize cosine similarity to [0, 1]
# adjacency_matrix = (1 + adjacency_matrix) / 2

# # Set diagonal to zero (no self-loops)
# np.fill_diagonal(adjacency_matrix, 0)

# # Step 3: Create the graph using NetworKit
# num_nodes = adjacency_matrix.shape[0]

In [ ]:
# coo = coo_matrix(adjacency_matrix).astype(np.float64)
# G = nk.graph.GraphFromCoo(coo, weighted=True)

In [ ]:
# G.weight(0,1)

In [ ]:
# plmCommunities = nk.community.detectCommunities(G, algo=nk.community.PLM(G), inspect=False)

In [ ]:
# type(plmCommunities)

In [ ]:
# nk.community.Modularity().getQuality(plmCommunities, G)

In [ ]:
def run_experiments(node_vectors, threshold_list, num_iterations=1, resolution=1):
    """
    Runs experiments using NetworKit's PLM Louvain algorithm and records results.

    Parameters:
    - node_vectors (torch.Tensor): Pre-processed tensor (node vectors).
    - threshold_list (list): List of threshold percentages to filter on (0.05, 0.1, etc.).
    - num_iterations (int): Number of times to run the experiment to ensure time stability.

    Returns:
    - pd.DataFrame: DataFrame with experiment results for each threshold.
    """
    results = []

    # Iterate through each threshold
    for threshold in threshold_list:
        row_result = {'threshold': threshold}

        # Step 1: Select top percentage of rows based on threshold
        num_rows = int(node_vectors.size(0) * threshold)
        sub_node_vectors = node_vectors[:num_rows, :].cpu().numpy()  # Selecting top rows
        row_result['num_nodes'] = num_rows

        # Check if the number of rows exceeds 25,000
        if sub_node_vectors.shape[0] > 25000:
            # Record 0 for time and modularity and skip execution
            row_result['networkit_time'] = 0
            row_result['networkit_modularity'] = 0
        else:
            # Step 2 and onward: Initialize timing just before computing norms and adjacency matrix
            nk_times = []
            nk_modularities = []
            for _ in range(num_iterations):
                start_time = time.time()  # Start the time before norms and adjacency matrix

                # Step 2: Compute the adjacency matrix (Cosine similarity)
                norms = np.linalg.norm(sub_node_vectors, axis=1)
                adjacency_matrix = np.dot(sub_node_vectors, sub_node_vectors.T) / (norms[:, None] * norms[None, :])
                adjacency_matrix = (1 + adjacency_matrix) / 2  # Normalize cosine similarity
                np.fill_diagonal(adjacency_matrix, 0)  # Set diagonal to zero (no self-loops)

                # Step 3: Create the graph using NetworKit from COO matrix
                coo = coo_matrix(adjacency_matrix).astype(np.float64)
                del adjacency_matrix, norms  # Free up memory
                gc.collect()  # Run garbage collection

                G = nk.graph.GraphFromCoo(coo, weighted=True)

                # Step 4: Run Louvain algorithm using PLM in NetworKit
                plmCommunities = nk.community.detectCommunities(G, algo=nk.community.PLM(G), inspect=False)
                nk_modularity = nk.community.Modularity().getQuality(plmCommunities, G)
                nk_time = time.time() - start_time  # Stop the time after the modularity is calculated

                # Append results for averaging later
                nk_times.append(nk_time)
                nk_modularities.append(nk_modularity)

                # Step 5: Free up memory and perform garbage collection
                del plmCommunities, coo, G
                gc.collect()

            # Record NetworKit Louvain results
            row_result['networkit_time'] = np.mean(nk_times)
            row_result['networkit_modularity'] = np.mean(nk_modularities)

        # Append row to results
        results.append(row_result)

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    return results_df

In [ ]:
l = [(i+1)/20 for i in range(20)]
l.sort()
l

[0.05,
 0.1,
 0.15,
 0.2,
 0.25,
 0.3,
 0.35,
 0.4,
 0.45,
 0.5,
 0.55,
 0.6,
 0.65,
 0.7,
 0.75,
 0.8,
 0.85,
 0.9,
 0.95,
 1.0]

In [ ]:
run_experiments(Flickr_data.data.x[~is_row_zero], l, 5)

Communities detected in 0.36222 [s]
Communities detected in 0.42204 [s]
Communities detected in 0.69577 [s]
Communities detected in 0.33305 [s]
Communities detected in 0.31775 [s]
Communities detected in 2.23480 [s]
Communities detected in 2.32262 [s]
Communities detected in 3.26455 [s]
Communities detected in 2.23886 [s]
Communities detected in 2.45100 [s]
Communities detected in 4.08268 [s]
Communities detected in 4.39601 [s]
Communities detected in 4.07386 [s]
Communities detected in 4.05891 [s]
Communities detected in 4.88597 [s]
Communities detected in 7.71025 [s]
Communities detected in 11.65292 [s]
Communities detected in 7.81955 [s]
Communities detected in 7.41910 [s]
Communities detected in 8.32954 [s]
Communities detected in 13.37971 [s]
Communities detected in 15.62575 [s]
Communities detected in 19.68158 [s]
Communities detected in 13.67882 [s]
Communities detected in 17.85552 [s]


,threshold,num_nodes,networkit_time,networkit_modularity
0,0.05,4462,5.108335,0.016856
1,0.10,8924,28.727172,0.016782
2,0.15,13387,71.923140,0.016567
3,0.20,17849,158.595530,0.016660
4,0.25,22312,281.757620,0.016576
5,0.30,26774,0.000000,0.000000
6,0.35,31237,0.000000,0.000000
7,0.40,35699,0.000000,0.000000
8,0.45,40162,0.000000,0.000000
9,0.50,44624,0.000000,0.000000


## AmazonProducts

In [ ]:
from torch_geometric.datasets import AmazonProducts

In [ ]:
AmazonProducts_data = AmazonProducts(root='data/AmazonProducts')

Processing...
Done!


In [ ]:
AmazonProducts_data.data

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


Data(x=[1569960, 200], edge_index=[2, 264339468], y=[1569960, 107], train_mask=[1569960], val_mask=[1569960], test_mask=[1569960])

In [ ]:
AmazonProducts_data.data['x']

tensor([[-0.1466,  0.2226, -0.3597,  ...,  0.1699,  0.8974,  1.6527],
        [-0.2805,  0.0190,  0.4301,  ..., -1.1758, -1.8365, -1.1693],
        [ 0.2554,  0.2519, -0.0291,  ...,  1.3751, -0.0735,  0.6262],
        ...,
        [-0.8121,  0.3626, -0.7781,  ...,  0.0639,  0.8645,  0.0389],
        [ 1.5977, -2.3989, -0.0569,  ..., -1.4413,  0.2966,  0.0985],
        [-0.1663,  0.0629, -0.0474,  ...,  0.1853, -0.1216, -0.9181]])

In [ ]:
Amazon_is_row_zero = (AmazonProducts_data.data.x == 0).all(dim=1)
AmazonProducts_data[Amazon_is_row_zero]

AmazonProducts()

In [ ]:
l = [0.002, 0.004, 0.006, 0.008, 0.01]

In [ ]:
run_experiments(AmazonProducts_data.data.x, l, 5)

Communities detected in 0.74755 [s]
Communities detected in 0.83538 [s]
Communities detected in 0.75052 [s]
Communities detected in 0.74934 [s]
Communities detected in 0.74345 [s]
Communities detected in 3.03848 [s]
Communities detected in 3.13484 [s]
Communities detected in 2.88294 [s]
Communities detected in 2.55042 [s]
Communities detected in 2.89687 [s]
Communities detected in 6.65157 [s]
Communities detected in 6.70212 [s]
Communities detected in 6.56652 [s]
Communities detected in 6.58865 [s]
Communities detected in 6.66914 [s]
Communities detected in 11.69085 [s]
Communities detected in 13.58848 [s]
Communities detected in 11.60764 [s]
Communities detected in 11.56044 [s]
Communities detected in 11.61105 [s]
Communities detected in 18.76984 [s]
Communities detected in 18.18742 [s]
Communities detected in 18.77526 [s]
Communities detected in 18.34014 [s]
Communities detected in 18.04608 [s]


,threshold,num_nodes,networkit_time,networkit_modularity
0,0.002,3139,2.948247,0.015533
1,0.004,6279,12.370117,0.014796
2,0.006,9419,32.627847,0.014637
3,0.008,12559,68.674507,0.014516
4,0.010,15699,127.554082,0.014349


In [ ]:
run_experiments(AmazonProducts_data.data.x, [0.5, 1], 1)

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


,threshold,num_nodes,networkit_time,networkit_modularity
0,0.5,784980,0,0
1,1.0,1569960,0,0


# Yelp

In [ ]:
from torch_geometric.datasets import Yelp

In [ ]:
Yelp_data = Yelp(root='data/Yelp')

Processing...
Done!


In [ ]:
Yelp_data.data.x.shape

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


torch.Size([716847, 300])

In [ ]:
Yelp_is_row_zero = (Yelp_data.data.x == 0).all(dim=1)
Yelp_data.data.x[Yelp_is_row_zero].shape

torch.Size([70, 300])

In [ ]:
l = [0.004, 0.008, 0.012, 0.016, 0.02]

In [ ]:
run_experiments(Yelp_data.data.x[~Yelp_is_row_zero], l, 5)

Communities detected in 0.14982 [s]
Communities detected in 0.14647 [s]
Communities detected in 0.16480 [s]
Communities detected in 0.17257 [s]
Communities detected in 0.15801 [s]
Communities detected in 0.58144 [s]
Communities detected in 0.56870 [s]
Communities detected in 0.63658 [s]
Communities detected in 0.76471 [s]
Communities detected in 0.57888 [s]
Communities detected in 1.29192 [s]
Communities detected in 1.29155 [s]
Communities detected in 1.08863 [s]
Communities detected in 1.26534 [s]
Communities detected in 1.39904 [s]
Communities detected in 2.07021 [s]
Communities detected in 1.96064 [s]
Communities detected in 2.49389 [s]
Communities detected in 2.00326 [s]
Communities detected in 2.28081 [s]
Communities detected in 3.42580 [s]
Communities detected in 3.48669 [s]
Communities detected in 3.59276 [s]
Communities detected in 3.17630 [s]
Communities detected in 3.23242 [s]


,threshold,num_nodes,networkit_time,networkit_modularity
0,0.004,2867,2.034827,0.003569
1,0.008,5734,8.328667,0.003675
2,0.012,8601,21.108823,0.003770
3,0.016,11468,50.028314,0.003789
4,0.020,14335,77.489891,0.003813


In [ ]:
run_experiments(Yelp_data.data.x[~Yelp_is_row_zero], [0.5, 1], 1)

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


,threshold,num_nodes,networkit_time,networkit_modularity
0,0.5,358388,0,0
1,1.0,716777,0,0


# Taobao

In [ ]:
from torch_geometric.datasets import Taobao

In [ ]:
Taobao_data = Taobao(root='data/Taobao')

In [ ]:
u_t_i = Taobao_data.data['user', 'to', 'item']['edge_index']
i_t_c = Taobao_data.data['item', 'to', 'category']['edge_index']

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [ ]:
def join_relationships_count(type_a_to_intermediate_tensor, intermediate_to_type_b_tensor, type_a_name='type_A', type_b_name='type_B'):
    """
    Joins two torch tensors using an intermediate entity as the intermediary and counts the occurrences of each pair.

    Parameters:
    - type_a_to_intermediate_tensor (torch.Tensor): A tensor with shape [2, n] representing relationships between type A and intermediate.
    - intermediate_to_type_b_tensor (torch.Tensor): A tensor with shape [2, m] representing relationships between intermediate and type B.
    - type_a_name (str): Name of the first type (e.g., 'type_A').
    - type_b_name (str): Name of the second type (e.g., 'type_B').

    Returns:
    - pd.DataFrame: A DataFrame with columns 'type_A', 'type_B', and 'count'.
    """
    # Extract rows for easier understanding
    type_a_ids = type_a_to_intermediate_tensor[0]  # IDs for type A (e.g., authors)
    intermediate_ids_from_type_a = type_a_to_intermediate_tensor[1]  # intermediate IDs related to type A

    intermediate_ids_from_type_b = intermediate_to_type_b_tensor[0]  # intermediate IDs related to type B
    type_b_ids = intermediate_to_type_b_tensor[1]  # IDs for type B (e.g., venues)

    # Convert tensors to lists for easier manipulation
    type_a_ids = type_a_ids.tolist()
    intermediate_ids_from_type_a = intermediate_ids_from_type_a.tolist()
    intermediate_ids_from_type_b = intermediate_ids_from_type_b.tolist()
    type_b_ids = type_b_ids.tolist()

    # Create a dictionary that maps intermediate IDs to type B
    intermediate_to_type_b = {}
    for intermediate_id, type_b_id in zip(intermediate_ids_from_type_b, type_b_ids):
        if intermediate_id not in intermediate_to_type_b:
            intermediate_to_type_b[intermediate_id] = []
        intermediate_to_type_b[intermediate_id].append(type_b_id)

    # Create the joined data for type A and type B via intermediate
    joined_data = []
    for type_a_id, intermediate_id in zip(type_a_ids, intermediate_ids_from_type_a):
        if intermediate_id in intermediate_to_type_b:
            for type_b_id in intermediate_to_type_b[intermediate_id]:
                joined_data.append((type_a_id, type_b_id))

    # Convert to a DataFrame and count occurrences
    df = pd.DataFrame(joined_data, columns=[type_a_name, type_b_name])
    count_df = df.groupby([type_a_name, type_b_name]).size().reset_index(name='count')

    return count_df

taobao_df = join_relationships_count(u_t_i, i_t_c)

In [ ]:
def filter_by_threshold(df, threshold_percentage=1, column_to_filter=1):
    """
    Filter the DataFrame based on a count threshold applied to either the first or second column.

    Parameters:
    - df (pd.DataFrame): The input DataFrame with at least two columns.
    - threshold_percentage (float): The threshold percentage of the maximum count.
    - filter_on (str): Specify 'first' to filter based on the first column, or 'second' to filter based on the second column. Default is 'second'.

    Returns:
    - pd.DataFrame: A filtered DataFrame where entries in the specified column meet the count threshold.
    """

    # Calculate the total number of unique values in the selected column
    col_counts = df.iloc[:, column_to_filter].value_counts()

    # Calculate the maximum count (the most frequent value in the selected column)
    max_count = col_counts.max()
    print(max_count)
    # Calculate the threshold count
    threshold = max_count * (threshold_percentage / 100)

    # Filter the DataFrame based on the threshold
    filtered_values = col_counts[col_counts >= threshold].index
    filtered_df = df[df.iloc[:, column_to_filter].isin(filtered_values)]

    return filtered_df

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
def df_to_matrix(df):
    [c1, c2, c3] = df.columns.tolist()
    pivot_df = df.pivot(index=c1, columns=c2, values=c3)

    pivot_df.fillna(0, inplace=True)

    return pivot_df
def convert_to_tfidf(pivot_df):

    # Step 2: Apply TF-IDF transformation
    tfidf_transformer = TfidfTransformer()
    tfidf_matrix = tfidf_transformer.fit_transform(pivot_df)

    # Step 3: Convert the resulting matrix back to a DataFrame for easier analysis
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), index=pivot_df.index, columns=pivot_df.columns)

    return tfidf_df

In [ ]:
filtered_taobao_df = filter_by_threshold(taobao_df, 15)

417880


In [ ]:
filtered_taobao_df.type_B.nunique()

66

In [ ]:
pivot_df = df_to_matrix(filtered_taobao_df)
tfidf_df = convert_to_tfidf(pivot_df)
tfidf_matrix = torch.tensor(tfidf_df.values)

In [ ]:
tfidf_matrix

tensor([[0.4061, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.3960, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.8664, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
       dtype=torch.float64)

In [ ]:
l = [0.005, 0.01, 0.015, 0.02, 0.025, 0.03]

In [ ]:
run_experiments(tfidf_matrix, l, 5)

Communities detected in 0.52504 [s]
Communities detected in 0.75844 [s]
Communities detected in 0.66609 [s]
Communities detected in 0.57949 [s]
Communities detected in 0.78713 [s]
Communities detected in 4.50797 [s]
Communities detected in 3.33764 [s]
Communities detected in 6.48501 [s]
Communities detected in 3.66016 [s]
Communities detected in 3.07990 [s]
Communities detected in 11.20871 [s]
Communities detected in 8.88298 [s]
Communities detected in 8.19188 [s]
Communities detected in 8.90736 [s]
Communities detected in 11.34237 [s]
Communities detected in 15.75524 [s]
Communities detected in 15.66583 [s]
Communities detected in 19.14901 [s]
Communities detected in 10.99414 [s]
Communities detected in 15.84070 [s]
Communities detected in 19.26391 [s]
Communities detected in 26.09361 [s]
Communities detected in 25.90915 [s]
Communities detected in 19.15669 [s]
Communities detected in 16.10648 [s]


,threshold,num_nodes,networkit_time,networkit_modularity
0,0.005,4684,4.357559,0.020747
1,0.010,9369,20.196703,0.020145
2,0.015,14054,57.037491,0.021169
3,0.020,18738,137.292324,0.020476
4,0.025,23423,223.453132,0.021223
5,0.030,28108,0.000000,0.000000


In [ ]:
run_experiments(tfidf_matrix, [0.5, 1], 1)

,threshold,num_nodes,networkit_time,networkit_modularity
0,0.5,468473,0,0
1,1.0,936946,0,0
